# Lab Day 10: Finetune Qwen3-VL-4B (LoRA) để định vị hư hại mặt đường
Đọc `README.md` và `GUIDE.md` trước. Notebook này chỉ là **khung**: bạn tự viết mã theo pseudo code trong GUIDE.

## 0. Môi trường (Colab T4)

In [ ]:
# Runtime → Change runtime type → T4 GPU
import os, pathlib, subprocess
IS_KAGGLE = bool(os.environ.get('KAGGLE_KERNEL_RUN_TYPE')) or pathlib.Path('/kaggle/input').exists()
BASE = pathlib.Path('/kaggle/working' if IS_KAGGLE else '/content')
os.chdir(str(BASE))
REPO = BASE / 'K4-Track-4-Day10-VisionLanguageModel'
if not (REPO / 'scripts' / 'prepare_data.py').exists():
    subprocess.run(['git', 'clone', 'https://github.com/TheViet298/K4-Track-4-Day10-VisionLanguageModel.git', str(REPO)], check=True)
os.chdir(str(REPO))
print('Environment:', 'Kaggle' if IS_KAGGLE else 'Colab')
!nvidia-smi -L
!pip -q install -r requirements-colab.txt
!pip -q install --upgrade --no-cache-dir 'bitsandbytes>=0.46.1'
import bitsandbytes as bnb
print('bitsandbytes:', bnb.__version__)

In [ ]:
# Kaggle: dùng Colab Secrets (KAGGLE_USERNAME, KAGGLE_KEY). KHÔNG viết khoá vào notebook.
import os, shutil
if IS_KAGGLE:
    DRIVE_DIR = '/kaggle/working/day10_backup'
    os.makedirs(DRIVE_DIR, exist_ok=True)
    print('Kaggle mode: no Colab secrets or Drive mount needed')
else:
    from google.colab import userdata, drive
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
    assert os.environ['KAGGLE_USERNAME'] and os.environ['KAGGLE_KEY']
    drive.mount('/content/drive')
    DRIVE_DIR = '/content/drive/MyDrive/day10_backup'
    os.makedirs(DRIVE_DIR, exist_ok=True)
print('Backup directory:', DRIVE_DIR)

## 1. Dữ liệu: tải và chia split

In [ ]:
from pathlib import Path
if IS_KAGGLE:
    attached = list(Path('/kaggle/input').rglob('labels-YOLO'))
    if attached:
        dataset_source = str(attached[0].parent)
        print('Using attached dataset:', dataset_source)
    else:
        try:
            import kagglehub
        except ImportError:
            subprocess.run(['pip', '-q', 'install', 'kagglehub'], check=True)
            import kagglehub
        dataset_source = kagglehub.dataset_download('lorenzoarcioni/road-damage-dataset-potholes-cracks-and-manholes')
        print('Dataset downloaded to:', dataset_source)
    subprocess.run(['python', 'scripts/prepare_data.py', '--source', dataset_source], check=True)
else:
    subprocess.run(['python', 'scripts/prepare_data.py', '--kaggle'], check=True)
assert Path('data/splits/meta.json').exists()
print('Dataset ready')

In [ ]:
import json, collections
def read_jsonl(path):
    with open(path, encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]
for split in ('train', 'val', 'test'):
    rows = read_jsonl(f'data/splits/{split}.jsonl')
    counts = collections.Counter(o['label'] for r in rows for o in r['objects'])
    print(split, 'images=', len(rows), 'boxes=', sum(counts.values()), dict(counts))

## 2. Nhãn: YOLO → JSON thang 0–1000

In [ ]:
import json
def yolo_to_box(label, yolo):
    xc, yc, w, h = yolo
    vals = [(xc-w/2)*1000, (yc-h/2)*1000, (xc+w/2)*1000, (yc+h/2)*1000]
    return {'label': label, 'bbox_2d': [max(0, min(1000, round(v))) for v in vals]}
def objects_to_answer(objects):
    boxes = [yolo_to_box(o['label'], o['yolo']) for o in objects]
    boxes.sort(key=lambda b: (b['bbox_2d'][1], b['bbox_2d'][0]))
    return json.dumps(boxes, separators=(',', ':'))
print(objects_to_answer([{'label':'pothole','yolo':[.5,.5,.2,.1]}]))

## 3. E0: zero-shot

In [ ]:
import os, json, torch
from PIL import Image
from tqdm.auto import tqdm
from transformers import AutoProcessor, BitsAndBytesConfig
try:
    from transformers import Qwen3VLForConditionalGeneration
except ImportError:
    from transformers import Qwen2_5_VLForConditionalGeneration as Qwen3VLForConditionalGeneration
MODEL_ID = 'Qwen/Qwen3-VL-4B-Instruct'
PROMPT = ('Detect road damage in this image. Return ONLY a JSON list. Classes are pothole, crack, manhole. ' +
          'Every bbox_2d coordinate must be an integer on the 0-1000 normalized scale. ' +
          'Format: [{\"label\":\"pothole\",\"bbox_2d\":[x1,y1,x2,y2]}].')
quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True, llm_int8_skip_modules=['visual','lm_head'])
processor = AutoProcessor.from_pretrained(MODEL_ID)
processor.tokenizer.padding_side = 'left'
model = Qwen3VLForConditionalGeneration.from_pretrained(MODEL_ID, quantization_config=quant, torch_dtype=torch.float16, device_map='auto')
model.eval()
test_rows = read_jsonl('data/splits/test.jsonl')
os.makedirs('work', exist_ok=True)
BACKUP_PATH = os.path.join(DRIVE_DIR, 'e0.jsonl')
if os.path.exists(BACKUP_PATH) and not os.path.exists('work/e0.jsonl'):
    shutil.copy2(BACKUP_PATH, 'work/e0.jsonl')
    print('Restored E0 from Drive backup')
done = set()
if os.path.exists('work/e0.jsonl'):
    done = {json.loads(line)['image_id'] for line in open('work/e0.jsonl', encoding='utf-8') if line.strip()}
remaining = [row for row in test_rows if row['image_id'] not in done][:20]
print(f'Resuming E0: {len(done)} done, this batch has {len(remaining)} images')
with open('work/e0.jsonl', 'a', encoding='utf-8') as out:
    for row in tqdm(remaining, desc='E0'):
        image = Image.open(row['image_path'] if os.path.exists(row['image_path']) else os.path.join('data', row['image_path'])).convert('RGB')
        messages = [{'role':'user','content':[{'type':'image','image':image},{'type':'text','text':PROMPT}]}]
        inputs = processor.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors='pt')
        inputs = {k: v.to(model.device) if hasattr(v, 'to') else v for k,v in inputs.items()}
        with torch.inference_mode():
            generated = model.generate(**inputs, max_new_tokens=128, do_sample=False)
        prompt_len = inputs['input_ids'].shape[1]
        raw = processor.batch_decode(generated[:, prompt_len:], skip_special_tokens=True)[0]
        out.write(json.dumps({'image_id':row['image_id'], 'raw':raw}, ensure_ascii=False)+'\n')
        out.flush()
        shutil.copy2('work/e0.jsonl', BACKUP_PATH)
print('Wrote and backed up work/e0.jsonl; run this cell again for the next batch.')

In [ ]:
!python scripts/score.py --pred work/e0.jsonl --name E0 --out report/evidence/e0.json

## 4. Dựng dữ liệu huấn luyện: chat template và mask loss

In [ ]:
# Training utilities: assistant-only loss masking and single-example collate.
def make_messages(row, answer=None):
    path = row['image_path'] if os.path.exists(row['image_path']) else os.path.join('data', row['image_path'])
    image = Image.open(path).convert('RGB')
    user = {'role':'user','content':[{'type':'image','image':image},{'type':'text','text':PROMPT}]}
    if answer is None: return [user]
    return [user, {'role':'assistant','content':[{'type':'text','text':answer}]}]
def encode_example(row):
    answer = objects_to_answer(row['objects'])
    prompt = processor.apply_chat_template(make_messages(row), tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors='pt')
    full = processor.apply_chat_template(make_messages(row, answer), tokenize=True, add_generation_prompt=False, return_dict=True, return_tensors='pt')
    n = prompt['input_ids'].shape[1]
    labels = full['input_ids'].clone()
    labels[:, :n] = -100
    full['labels'] = labels
    return full
print('Training example ready; answer tokens are masked correctly.')

## 5. Setup LoRA / đóng băng cho E1, E2, E3

In [ ]:
from peft import LoraConfig, get_peft_model
def apply_setup(model, setup):
    for p in model.parameters(): p.requires_grad = False
    model.enable_input_require_grads()
    # Keep suffix targets; full dotted names can wrap unintended modules.
    targets = ['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj']
    if setup == 'E3': targets += ['qkv','proj','linear_fc1','linear_fc2']
    cfg = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM', target_modules=targets)
    peft_model = get_peft_model(model, cfg)
    if setup in ('E2','E3'):
        for name, p in peft_model.named_parameters():
            if '.visual.merger.' in name or '.visual.deepstack_merger_list.' in name:
                p.data = p.data.float(); p.requires_grad = True
    peft_model.print_trainable_parameters()
    return peft_model
print('apply_setup ready')

## 6. Vòng huấn luyện

In [ ]:
# Training loop: balanced random sampling + 4-step gradient accumulation.
import time, random, torch
from torch.nn.utils import clip_grad_norm_
train_rows = read_jsonl('data/splits/train.jsonl')
# Chọn setup cần chạy: E1, E2 hoặc E3. Hiện tại tiếp tục với E2.
TRAIN_SETUP = 'E2'
TRAIN_STEPS = 300
GRAD_ACCUM = 4
peft_model = apply_setup(model, TRAIN_SETUP)
peft_model.config.use_cache = False
try: peft_model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
except TypeError: peft_model.gradient_checkpointing_enable()
trainable = [p for p in peft_model.parameters() if p.requires_grad]
lr = 1e-5 if TRAIN_SETUP in ('E2','E3') else 2e-4
opt = torch.optim.AdamW(trainable, lr=lr, weight_decay=0.0)
rng = random.Random(10)
class_rows = {c:[r for r in train_rows if any(o['label']==c for o in r['objects'])] for c in ('pothole','crack','manhole')}
start = time.time(); peft_model.train(); opt.zero_grad(set_to_none=True)
for step in range(TRAIN_STEPS):
    losses = []
    for micro in range(GRAD_ACCUM):
        cls = ('pothole','crack','manhole')[(step*GRAD_ACCUM+micro) % 3]
        row = rng.choice(class_rows[cls])
        batch = encode_example(row)
        batch = {k: v.to(peft_model.device) if hasattr(v, 'to') else v for k,v in batch.items()}
        with torch.autocast('cuda', dtype=torch.float16): loss = peft_model(**batch).loss / GRAD_ACCUM
        if not torch.isfinite(loss): raise RuntimeError(f'non-finite loss at step {step}: {loss}')
        loss.backward(); losses.append(loss.item() * GRAD_ACCUM)
    clip_grad_norm_(trainable, 1.0)
    opt.step(); opt.zero_grad(set_to_none=True)
    if (step+1) % 10 == 0: print(f'step {step+1}/{TRAIN_STEPS} loss={sum(losses)/len(losses):.4f}')
adapter_dir = f'work/adapters/{TRAIN_SETUP.lower()}'
os.makedirs(adapter_dir, exist_ok=True)
peft_model.save_pretrained(adapter_dir)
run = {'setup':TRAIN_SETUP,'trainable_params':sum(p.numel() for p in trainable),'train_minutes':(time.time()-start)/60,'peak_vram_gb':torch.cuda.max_memory_allocated()/1024**3}
json.dump(run, open(os.path.join(adapter_dir, 'run.json'),'w'), indent=2)
print(f'Saved {TRAIN_SETUP}:', run)

## 7. Suy luận và chấm E1–E3

In [ ]:
# Inference and scoring for the currently trained setup
EVAL_SETUP = TRAIN_SETUP
eval_name = EVAL_SETUP.lower()
peft_model.eval()
with open(f'work/{eval_name}.jsonl','w',encoding='utf-8') as out:
    for row in tqdm(test_rows, desc=EVAL_SETUP):
        messages = make_messages(row)
        inputs = processor.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors='pt')
        inputs = {k:v.to(peft_model.device) if hasattr(v,'to') else v for k,v in inputs.items()}
        with torch.inference_mode(): generated = peft_model.generate(**inputs, max_new_tokens=128, do_sample=False)
        n = inputs['input_ids'].shape[1]
        raw = processor.batch_decode(generated[:,n:], skip_special_tokens=True)[0]
        out.write(json.dumps({'image_id':row['image_id'],'raw':raw}, ensure_ascii=False)+'\n')
run = json.load(open(os.path.join('work', 'adapters', eval_name, 'run.json')))
subprocess.run(['python','scripts/score.py','--pred',f'work/{eval_name}.jsonl','--name',EVAL_SETUP,'--out',f'report/evidence/{eval_name}.json','--meta',f"trainable_params={run['trainable_params']}",'--meta',f"peak_vram_gb={run['peak_vram_gb']}",'--meta',f"train_minutes={run['train_minutes']}"], check=True)

## 8. E4 (ablation tự chọn) và E5 (tuỳ chọn, GRPO)

In [ ]:
# E4 configuration: run one controlled ablation against the best SFT setup.
# The default comparison is LoRA rank; change RANK and rerun the training/inference cells.
E4_RANKS = [4, 64]
print('E4 rank ablation configured:', E4_RANKS)
print('For each value set LoraConfig(r=rank, lora_alpha=2*rank), train 300 steps, score test, and save e4_rank_<rank>.json.')